# Module 05 — Lesson 09
# Apply (Custom Functions)

---

> Every operation so far -- filtering, sorting, grouping, merging -- has been a built-in pandas tool doing something generic. But sometimes the logic you need is genuinely custom: "label each bill as Low/Medium/High," "combine three columns into one description string." `.apply()` is the escape hatch that lets you run *your own function* across a Series or DataFrame -- but it's also the tool most likely to be reached for when a faster, simpler vectorized operation would do the same job.

---
## 🎯 Learning Objectives

By the end of this lesson, you will be able to:

- Use `Series.apply()` to run a custom function (or lambda) over every value in a column
- Use `DataFrame.apply(axis=1)` to run a function across each **row**, using multiple columns at once
- Use `Series.map()` to substitute values via a dictionary
- Explain why vectorized operations are usually faster than `.apply()`, and when `.apply()` is genuinely needed
- Avoid the classic `axis=0` vs `axis=1` mix-up in `DataFrame.apply()`

In [ ]:
import pandas as pd

tips = pd.read_csv("data/tips.csv")
print(tips.head())

---
## 1. `Series.apply()` — A Custom Function, One Value at a Time

`.apply()` on a Series calls your function once per value and collects the results into a new Series. Use it when the transformation isn't a simple arithmetic expression -- like sorting a number into a category.

In [ ]:
def categorize_bill(amount):
    if amount < 15:
        return "Low"
    elif amount < 30:
        return "Medium"
    else:
        return "High"

tips["bill_category"] = tips["total_bill"].apply(categorize_bill)
print(tips[["total_bill", "bill_category"]].head(8))

A `lambda` works too, for logic short enough to write inline -- no need to define a full function for a one-liner.

In [ ]:
tips["rounded_bill"] = tips["total_bill"].apply(lambda x: round(x))
print(tips[["total_bill", "rounded_bill"]].head(3))

---
## 2. `DataFrame.apply(axis=1)` — A Custom Function, One Row at a Time

Set `axis=1` to call your function once per **row**, passing the whole row (as a Series) so you can combine multiple columns. This is how you compute something like "tip as a percentage of the bill" when the logic needs more than one column.

In [ ]:
def tip_percentage(row):
    return round(row["tip"] / row["total_bill"] * 100, 1)

tips["tip_pct"] = tips.apply(tip_percentage, axis=1)
print(tips[["total_bill", "tip", "tip_pct"]].head(5))

---
## 3. `Series.map()` — Substituting Values via a Dictionary

`.map()` is the simplest way to translate every value in a Series through a lookup table -- shorter than `.apply()` with an if/elif chain when you're just renaming categories.

In [ ]:
day_full_names = {"Thur": "Thursday", "Fri": "Friday", "Sat": "Saturday", "Sun": "Sunday"}

tips["day_full"] = tips["day"].map(day_full_names)
print(tips[["day", "day_full"]].drop_duplicates())

---
## 4. Why Vectorized Operations Usually Beat `.apply()`

Pandas (via NumPy) can perform arithmetic on an *entire column at once* -- that's what "vectorized" means. `.apply()` instead calls a plain Python function once per row, which is far slower at scale. If your logic is just arithmetic or comparisons, write it directly instead of wrapping it in `.apply()`.

In [ ]:
import time

# The SAME calculation, done two ways: vectorized vs .apply()
start = time.perf_counter()
vectorized_result = tips["tip"] / tips["total_bill"] * 100
vectorized_time = time.perf_counter() - start

start = time.perf_counter()
apply_result = tips.apply(lambda row: row["tip"] / row["total_bill"] * 100, axis=1)
apply_time = time.perf_counter() - start

print(f"Vectorized time: {vectorized_time * 1000:.3f} ms")
print(f"apply() time:    {apply_time * 1000:.3f} ms")
print(f"apply() was ~{apply_time / vectorized_time:.0f}x slower on this {tips.shape[0]}-row DataFrame")
print(f"Results match: {vectorized_result.round(5).equals(apply_result.round(5))}")

---
## ⚠️ Common Mistakes

In [ ]:
# -- Mistake 1: Reaching for apply() when a vectorized op already works ---------

print("Mistake 1 — the timing comparison above already proved this, but to say it")
print("plainly: if your logic is arithmetic or a comparison, don't wrap it in")
print("apply() at all:")
print("  SLOW:  tips.apply(lambda row: row['tip'] / row['total_bill'] * 100, axis=1)")
print("  FAST:  tips['tip'] / tips['total_bill'] * 100")

In [ ]:
# -- Mistake 2: Forgetting axis=1 for row-wise logic ----------------------------

print("Mistake 2 — DataFrame.apply() defaults to axis=0 (column-wise), not row-wise:")
try:
    # No axis=1 -- pandas applies tip_percentage() to each COLUMN (a whole Series),
    # not each row, so row['tip'] inside the function makes no sense here
    tips.apply(tip_percentage)
except Exception as e:
    print(f"  Caught error: {type(e).__name__}: {e}")
print()
print("  Correct: pass axis=1 explicitly whenever your function expects a single row:")
print(tips.apply(tip_percentage, axis=1).head(3))

In [ ]:
# -- Mistake 3: .map() with a dict that doesn't cover every value ---------------

print("Mistake 3 — any value missing from the mapping dict silently becomes NaN:")
incomplete_map = {"Thur": "Thursday", "Fri": "Friday"}   # 'Sat' and 'Sun' missing on purpose

mapped = tips["day"].map(incomplete_map)
print(f"  Missing values after incomplete map: {mapped.isna().sum()}")
print(f"  (out of {tips.shape[0]} rows -- every Sat/Sun row silently became NaN)")
print()
print("  Correct: make sure the dict covers every possible value, or fall back")
print("  to the original value for anything unmapped:")
safe_mapped = tips["day"].map(incomplete_map).fillna(tips["day"])
print(f"  Missing values after fillna() fallback: {safe_mapped.isna().sum()}")

---
## ✏️ Practice Exercises

### Exercise 1 — Starter: `Series.apply()`

Write a function `size_label(n)` that returns `"Small"` for `size <= 2`, `"Medium"` for `size` 3 or 4, and `"Large"` otherwise. Apply it to `tips["size"]` and store the result in a new column `"party_size_label"`.

In [ ]:
# Your code here

### Exercise 2 — `DataFrame.apply(axis=1)`

Write a row-wise function that returns `"Generous"` if `tip / total_bill > 0.2`, and `"Typical"` otherwise. Apply it with `axis=1` and store the result in a new column `"tipping_style"`.

In [ ]:
# Your code here

### Exercise 3 — `.map()`

Create a dictionary mapping `"Male"` → `"M"` and `"Female"` → `"F"`. Use `.map()` on `tips["sex"]` to create a new column `"sex_code"`.

In [ ]:
# Your code here

### Exercise 4 — Vectorize It

Rewrite this `.apply()` call as a plain vectorized expression (no `apply()` at all), and confirm both give the same result with `.equals()`:

```python
tips.apply(lambda row: row["total_bill"] + row["tip"], axis=1)
```

In [ ]:
# Your code here

### Exercise 5 — (Challenge) Row-Wise Label from Multiple Columns

Write a row-wise function that returns `"Big Dinner"` if `time == "Dinner"` **and** `size >= 4`, and `"Other"` otherwise. Apply it with `axis=1`, store it in `"occasion"`, then use `.value_counts()` to see how many rows fall into each category.

In [ ]:
# Your code here

---
## 📌 Key Takeaways

- **`.apply()` runs a custom function over a Series (one value at a time) or a DataFrame (`axis=1` for row-wise, `axis=0`/default for column-wise)** — reach for it when your logic genuinely can't be written as simple vectorized arithmetic or comparisons.

- **Vectorized operations are almost always faster than `.apply()`** — the timing comparison in this lesson showed a real, measurable gap on just 244 rows; that gap only grows on larger datasets.

- **`.map()` is the simplest way to substitute values in a Series via a dictionary** — but any value missing from the dict silently becomes `NaN`, so make sure your mapping is complete or add a `.fillna()` fallback.

---
## What's Next?

**Lesson 10 — Pivot Tables** reshapes a DataFrame into a summary grid — rows and columns both become categories, with an aggregation filling every cell.